# ALD Showerhead — MultiHead MeshGraphNet Retraining
**83 reactingFoam cases · Colab A100 · hidden=256 / 15 layers**

**Before running:**
1. Runtime → Change runtime type → **A100 GPU**
2. Upload `data/processed/ald_hdf5/` from your Mac to `MyDrive/cfd-ald-app/ald_hdf5/`
3. Runtime → **Run all**


In [ ]:
import torch, subprocess, sys

assert torch.cuda.is_available(), 'Switch runtime to A100 GPU first!'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.', '')
cmd = (
    'pip install -q torch-geometric torch-scatter torch-sparse torch-cluster '
    f'-f https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
)
print('Installing torch_geometric ...')
subprocess.run(cmd, shell=True, check=True)
print('Done.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR   = DRIVE_BASE / 'ald_hdf5'
CKPT_DIR   = DRIVE_BASE / 'checkpoints' / 'multihead'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Verify Drive is writable before anything else
_t = CKPT_DIR / '_test.txt'
_t.write_text('ok')
assert _t.exists() and '/drive/' in str(_t), 'Drive not writable — check mount!'
_t.unlink()
print('Drive write: OK ->', CKPT_DIR)

assert HDF5_DIR.exists(), f'HDF5 folder not found: {HDF5_DIR}'
print('HDF5 files found:', len(sorted(HDF5_DIR.glob('*.h5'))))

CFG = dict(
    hidden_dim      = 256,
    n_layers        = 15,
    k_neighbors     = 6,
    node_input_dim  = 22,
    edge_input_dim  = 4,
    flow_out_dim    = 4,
    heat_out_dim    = 1,
    species_out_dim = 1,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    epochs          = 150,
    save_every      = 5,
    n_train_nodes   = 4096,
    n_val_nodes     = 4096,
)

DEVICE      = torch.device('cuda')
OUTPUT_COLS = ['Ux', 'Uy', 'Uz', 'p', 'T', 'TMA']
print('Config:', CFG)


In [ ]:
import numpy as np
import h5py, time, shutil, os
import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm, trange
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')


In [ ]:
LOCAL_HDF5 = Path('/content/ald_hdf5')
LOCAL_HDF5.mkdir(exist_ok=True)

print('Copying HDF5 files from Drive to local disk ...')
all_h5 = sorted(HDF5_DIR.glob('*.h5'))
for f in tqdm(all_h5):
    dst = LOCAL_HDF5 / f.name
    if not dst.exists():
        shutil.copy2(f, dst)

val_files   = sorted(LOCAL_HDF5.glob('cfd_val_*.h5'))
train_files = sorted(LOCAL_HDF5.glob('case_*.h5'))
print('Train:', len(train_files), ' Val:', len(val_files))


In [ ]:
def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods += [nn.SiLU(), nn.LayerNorm(dims[i+1])]
    return nn.Sequential(*mods)


class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3 * hidden, hidden)
        self.node_mlp  = mlp(2 * hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)

    def forward(self, x, edge_index, edge_attr):
        row, col  = edge_index
        e_in      = torch.cat([x[row], x[col], edge_attr], dim=-1)
        edge_attr = self.edge_norm(self.edge_mlp(e_in) + edge_attr)
        N   = x.size(0)
        agg = self.propagate(edge_index, x=x, edge_attr=edge_attr, size=(N, N))
        x   = self.node_norm(self.node_mlp(torch.cat([x, agg], dim=-1)) + x)
        return x, edge_attr

    def message(self, edge_attr):
        return edge_attr


class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim,
                 flow_out=4, heat_out=1, species_out=1,
                 hidden=256, n_layers=15):
        super().__init__()
        self.node_enc    = mlp(node_dim, hidden)
        self.edge_enc    = mlp(edge_dim, hidden)
        self.processors  = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)

    def forward(self, x, edge_index, edge_attr):
        x  = self.node_enc(x)
        ea = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, ea = proc(x, edge_index, ea)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)


model = MultiHeadMGN(
    node_dim    = CFG['node_input_dim'],
    edge_dim    = CFG['edge_input_dim'],
    flow_out    = CFG['flow_out_dim'],
    heat_out    = CFG['heat_out_dim'],
    species_out = CFG['species_out_dim'],
    hidden      = CFG['hidden_dim'],
    n_layers    = CFG['n_layers'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print('MultiHeadMGN params:', round(n_params/1e6, 1), 'M')


In [ ]:
def load_case(h5_path):
    with h5py.File(h5_path, 'r') as h:
        coords = h['coords'][:]
        nf     = h['inputs/node_features'][:]
        gf     = h['inputs/global'][:]
        y      = h['outputs/node_fields'][:]
    N  = min(len(coords), len(nf), len(y))
    xi = np.concatenate([nf[:N], np.tile(gf, (N, 1))], axis=1)
    return {'coords': coords[:N], 'x': xi, 'y': y[:N]}


def build_graph(case, n_nodes, k, device):
    coords = case['coords']
    x      = case['x']
    y      = case['y']
    N_full = len(coords)
    if n_nodes < N_full:
        idx    = np.random.choice(N_full, n_nodes, replace=False)
        coords = coords[idx]; x = x[idx]; y = y[idx]
    N       = len(coords)
    tree    = cKDTree(coords)
    _, nbrs = tree.query(coords, k=k+1)
    nbrs    = nbrs[:, 1:]
    src     = np.repeat(np.arange(N), k)
    dst     = nbrs.flatten()
    diff    = coords[dst] - coords[src]
    dist    = np.linalg.norm(diff, axis=1, keepdims=True)
    med     = float(np.median(dist)) + 1e-8
    ef      = np.concatenate([diff/med, dist/med], axis=1).astype(np.float32)
    return {
        'x':          torch.tensor(x,  dtype=torch.float32).to(device),
        'edge_index': torch.tensor(np.stack([src, dst]), dtype=torch.long).to(device),
        'edge_attr':  torch.tensor(ef, dtype=torch.float32).to(device),
        'y':          torch.tensor(y,  dtype=torch.float32).to(device),
    }


print('Pre-loading cases into RAM ...')
train_cases = [load_case(f) for f in tqdm(train_files)]
val_cases   = [load_case(f) for f in val_files]
print('Loaded. Mean nodes/case:', int(np.mean([len(c['coords']) for c in train_cases])))


In [ ]:
all_x = np.concatenate([c['x'] for c in train_cases], axis=0)
all_y = np.concatenate([c['y'] for c in train_cases], axis=0)

node_mean = all_x.mean(axis=0).astype(np.float32)
node_std  = (all_x.std(axis=0) + 1e-8).astype(np.float32)
out_mean  = all_y.mean(axis=0).astype(np.float32)
out_std   = (all_y.std(axis=0) + 1e-8).astype(np.float32)

NORM = {
    'node_mean':   node_mean.tolist(),
    'node_std':    node_std.tolist(),
    'out_mean':    out_mean.tolist(),
    'out_std':     out_std.tolist(),
    'output_cols': OUTPUT_COLS,
}

print('Output field stats:')
for col, m, s in zip(OUTPUT_COLS, out_mean, out_std):
    print(' ', col, ' mean=', round(float(m),4), ' std=', round(float(s),6))

del all_x


In [ ]:
K  = CFG['k_neighbors']
NT = CFG['n_train_nodes']
NV = CFG['n_val_nodes']

print('Pre-building k-NN graphs (once, reused every epoch) ...')
train_graphs = [build_graph(c, NT, K, DEVICE) for c in tqdm(train_cases)]
val_graphs   = [build_graph(c, NV, K, DEVICE) for c in val_cases]
print('Cached', len(train_graphs), 'train +', len(val_graphs), 'val graphs on GPU')


In [ ]:
_om = torch.tensor(out_mean, device=DEVICE)
_os = torch.tensor(out_std,  device=DEVICE)

def normalise_y(y):
    return (y - _om) / _os

def compute_loss(fp, hp, sp, y_norm):
    fl = nn.functional.mse_loss(fp,             y_norm[:, :4])
    hl = nn.functional.mse_loss(hp.squeeze(-1), y_norm[:,  4])
    sl = nn.functional.mse_loss(sp.squeeze(-1), y_norm[:,  5])
    return fl + hl + sl, {'flow': fl.item(), 'heat': hl.item(), 'species': sl.item()}


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 0.01
)
scaler = torch.cuda.amp.GradScaler()

start_epoch   = 0
train_losses  = []
val_losses    = []
head_logs     = {'flow': [], 'heat': [], 'species': []}

# Resume from latest Drive checkpoint if available
saved = sorted(CKPT_DIR.glob('multihead_epoch*.pt'))
if saved:
    ck = torch.load(saved[-1], map_location=DEVICE)
    model.load_state_dict(ck['model'])
    start_epoch  = ck['epoch'] + 1
    train_losses = ck.get('train_losses', [])
    val_losses   = ck.get('val_losses', [])
    head_logs    = ck.get('head_logs', head_logs)
    print('Resumed from epoch', start_epoch - 1)
else:
    print('Training from scratch. Epochs =', CFG['epochs'])


In [ ]:
model.train()
pbar = trange(start_epoch, CFG['epochs'], desc='Epoch')

for epoch in pbar:
    ep_loss  = 0.0
    ep_heads = {'flow': 0.0, 'heat': 0.0, 'species': 0.0}
    order    = np.random.permutation(len(train_graphs))
    optimizer.zero_grad()

    for step, idx in enumerate(order):
        g      = train_graphs[idx]
        y_norm = normalise_y(g['y'])

        with torch.cuda.amp.autocast():
            fp, hp, sp = model(g['x'], g['edge_index'], g['edge_attr'])
            loss, heads = compute_loss(fp, hp, sp, y_norm)
            loss = loss / len(order)

        scaler.scale(loss).backward()
        ep_loss += loss.item() * len(order)
        for k_head in heads:
            ep_heads[k_head] += heads[k_head] / len(order)

        if (step + 1) % 8 == 0 or (step + 1) == len(order):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    scheduler.step()
    train_losses.append(ep_loss)
    for k_head in ep_heads:
        head_logs[k_head].append(ep_heads[k_head])

    # Validate every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == start_epoch:
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for g in val_graphs:
                fp, hp, sp = model(g['x'], g['edge_index'], g['edge_attr'])
                vl += compute_loss(fp, hp, sp, normalise_y(g['y']))[0].item()
        val_losses.append(round(vl / len(val_graphs), 5))
        model.train()

    pbar.set_postfix({
        'tr':  round(ep_loss, 4),
        'val': val_losses[-1] if val_losses else '-',
        'lr':  round(scheduler.get_last_lr()[0], 6),
    })

    # Save checkpoint to Drive every save_every epochs
    if (epoch + 1) % CFG['save_every'] == 0:
        ck_path = CKPT_DIR / ('multihead_epoch' + str(epoch+1).zfill(4) + '.pt')
        torch.save({
            'model':        model.state_dict(),
            'cfg':          CFG,
            'norm':         NORM,
            'epoch':        epoch,
            'train_losses': train_losses,
            'val_losses':   val_losses,
            'head_logs':    head_logs,
        }, ck_path)
        print('  Saved', ck_path.name)

print('Training complete.')


In [ ]:
final_path = CKPT_DIR / 'multihead_final.pt'
torch.save({
    'model':             model.state_dict(),
    'cfg':               CFG,
    'norm':              NORM,
    'final_train_loss':  train_losses[-1],
    'final_val_loss':    val_losses[-1] if val_losses else None,
    'per_head_train':    {k: head_logs[k][-1] for k in head_logs},
    'n_cases':           len(train_cases),
    'data_source':       'reactingFoam_openfoam_80cases',
}, final_path)

size_mb = round(os.path.getsize(final_path) / 1e6, 1)
print('Saved:', final_path, '(' + str(size_mb) + ' MB)')
print('Final train loss:', round(train_losses[-1], 4))
print('Final val loss:  ', val_losses[-1] if val_losses else 'n/a')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='train')
if val_losses:
    axes[0].plot([i*10 for i in range(len(val_losses))], val_losses, label='val')
axes[0].set_title('Total loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].set_yscale('log')

for head in ('flow', 'heat', 'species'):
    axes[1].plot(head_logs[head], label=head)
axes[1].set_title('Per-head train loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'loss_curve.png'), dpi=150)
plt.show()


In [ ]:
ck    = torch.load(final_path, map_location='cpu')
cfg_c = ck['cfg']

m2 = MultiHeadMGN(
    node_dim    = cfg_c['node_input_dim'],
    edge_dim    = cfg_c['edge_input_dim'],
    flow_out    = cfg_c['flow_out_dim'],
    heat_out    = cfg_c['heat_out_dim'],
    species_out = cfg_c['species_out_dim'],
    hidden      = cfg_c['hidden_dim'],
    n_layers    = cfg_c['n_layers'],
)
m2.load_state_dict(ck['model'])
m2.eval()

g = val_graphs[0]
x_cpu  = g['x'].cpu()
ei_cpu = g['edge_index'].cpu()
ea_cpu = g['edge_attr'].cpu()

with torch.no_grad():
    fp, hp, sp = m2(x_cpu, ei_cpu, ea_cpu)

print('Checkpoint loads OK')
print('  flow    output:', fp.shape, ' range [', round(fp.min().item(),3), ',', round(fp.max().item(),3), ']')
print('  heat    output:', hp.shape, ' range [', round(hp.min().item(),3), ',', round(hp.max().item(),3), ']')
print('  species output:', sp.shape, ' range [', round(sp.min().item(),3), ',', round(sp.max().item(),3), ']')
print('  norm keys:', list(ck['norm'].keys()))
print('  cfg  keys:', list(cfg_c.keys()))
print('Ready to use in app.py')
